# Assignment 2
2025 | IN6227 Data Mining | Roshani Ayu Pranasti | G2504973A

## Install Orange Association Library

- Documentation: https://orange3-associate.readthedocs.io/en/latest/
- GitHub: https://github.com/biolab/orange3-associate/tree/master

In [1]:
!pip3 install orange3 orange3-associate

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


## Import Libraries

In [2]:
import numpy as np
import pandas as pd
import time
import warnings
from itertools import chain, combinations # For brute-force association rules mining
from orangecontrib.associate.fpgrowth import *  # Association rules mining in Orange 3

warnings.filterwarnings('ignore')

## Dataset Definition
I have prepared 1 dataset that is Bakery Sales Dataset. More about the dataset can be read on `README.md`.
1. **Bakery Sales Dataset**: `bakery_sales.csv`

With this dataset, I will further prepare several datasets varying sizes on the number of unique items, but the number of transactions does not change.

## Exploratory Data Analysis

In [3]:
# Load dataset
bakery_dataset = pd.read_csv("bakery_sales.csv")
display(bakery_dataset)

,Transaction,Item,date_time,period_day,weekday_weekend
0,1,Bread,10/30/2016 9:58,morning,weekend
1,2,Scandinavian,10/30/2016 10:05,morning,weekend
2,2,Scandinavian,10/30/2016 10:05,morning,weekend
3,3,Hot chocolate,10/30/2016 10:07,morning,weekend
4,3,Jam,10/30/2016 10:07,morning,weekend
...,...,...,...,...,...
20502,9682,Coffee,4/9/2017 14:32,afternoon,weekend
20503,9682,Tea,4/9/2017 14:32,afternoon,weekend
20504,9683,Coffee,4/9/2017 14:57,afternoon,weekend
20505,9683,Pastry,4/9/2017 14:57,afternoon,weekend


In [4]:
print("Number of attributes in dataset:", bakery_dataset.shape[1])
print("Number of data in dataset:", bakery_dataset.shape[0], "\n")
bakery_dataset.info()

Number of attributes in dataset: 5
Number of data in dataset: 20507 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20507 entries, 0 to 20506
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Transaction      20507 non-null  int64 
 1   Item             20507 non-null  object
 2   date_time        20507 non-null  object
 3   period_day       20507 non-null  object
 4   weekday_weekend  20507 non-null  object
dtypes: int64(1), object(4)
memory usage: 801.2+ KB


In [5]:
print("Number of unknown or missing values in dataset:")
bakery_dataset.isnull().sum()

Number of unknown or missing values in dataset:


Transaction        0
Item               0
date_time          0
period_day         0
weekday_weekend    0
dtype: int64

In [6]:
# Check unique values for each column
for column in bakery_dataset.columns:
    print(column, "\n", bakery_dataset[column].unique())

Transaction 
 [   1    2    3 ... 9682 9683 9684]
Item 
 ['Bread' 'Scandinavian' 'Hot chocolate' 'Jam' 'Cookies' 'Muffin' 'Coffee'
 'Pastry' 'Medialuna' 'Tea' 'Tartine' 'Basket' 'Mineral water'
 'Farm House' 'Fudge' 'Juice' "Ella's Kitchen Pouches" 'Victorian Sponge'
 'Frittata' 'Hearty & Seasonal' 'Soup' 'Pick and Mix Bowls' 'Smoothies'
 'Cake' 'Mighty Protein' 'Chicken sand' 'Coke' 'My-5 Fruit Shoot'
 'Focaccia' 'Sandwich' 'Alfajores' 'Eggs' 'Brownie' 'Dulce de Leche'
 'Honey' 'The BART' 'Granola' 'Fairy Doors' 'Empanadas' 'Keeping It Local'
 'Art Tray' 'Bowl Nic Pitt' 'Bread Pudding' 'Adjustment' 'Truffles'
 'Chimichurri Oil' 'Bacon' 'Spread' 'Kids biscuit' 'Siblings'
 'Caramel bites' 'Jammie Dodgers' 'Tiffin' 'Olum & polenta' 'Polenta'
 'The Nomad' 'Hack the stack' 'Bakewell' 'Lemon and coconut' 'Toast'
 'Scone' 'Crepes' 'Vegan mincepie' 'Bare Popcorn' 'Muesli' 'Crisps'
 'Pintxos' 'Gingerbread syrup' 'Panatone' 'Brioche and salami'
 'Afternoon with the baker' 'Salad' 'Chicken Stew'

In [7]:
# Check unique values in Transaction column
uniqe_transactions = bakery_dataset["Transaction"].unique()
last_transaction = uniqe_transactions[-1]
print("Unique transactions count:", len(uniqe_transactions))
print("Last transaction:", last_transaction)

# Check missing transactions
missing_transactions = [i for i in range(1, last_transaction + 1) if i not in uniqe_transactions]
print("Missing transactions count:", len(missing_transactions))
print("Missing transactions:\n", missing_transactions)

Unique transactions count: 9465
Last transaction: 9684
Missing transactions count: 219
Missing transactions:
 [53, 177, 273, 315, 362, 434, 472, 496, 561, 582, 607, 614, 619, 649, 679, 694, 731, 801, 823, 827, 1045, 1067, 1079, 1113, 1302, 1312, 1385, 1481, 1489, 1596, 1628, 1766, 1883, 1884, 1918, 2032, 2046, 2047, 2056, 2060, 2064, 2139, 2233, 2240, 2247, 2292, 2293, 2299, 2306, 2336, 2345, 2349, 2394, 2405, 2411, 2455, 2456, 2465, 2480, 2483, 2484, 2485, 2531, 2542, 2544, 2658, 2779, 2784, 2864, 2883, 2947, 2965, 2999, 3012, 3014, 3016, 3057, 3072, 3132, 3140, 3254, 3269, 3273, 3302, 3308, 3345, 3365, 3511, 3528, 3544, 3553, 3569, 3604, 3607, 3671, 3745, 3862, 3863, 3963, 3973, 4021, 4084, 4091, 4092, 4093, 4122, 4137, 4207, 4255, 4282, 4301, 4318, 4466, 4467, 4469, 4470, 4473, 4474, 4481, 4488, 4523, 4526, 4538, 4544, 4580, 4621, 4633, 4648, 4700, 4755, 4809, 4810, 4815, 4829, 4947, 4970, 5053, 5074, 5085, 5091, 5186, 5246, 5266, 5270, 5330, 5331, 5332, 5333, 5334, 5335, 5351, 5439

## Data Preprocessing

### Encode Transactions using One-Hot Encoding
For a simple association rules mining task, the goal is to find relationships between items within a transaction, regardless of when it happened. In this case, the time of day (`period_day`) or day of the week (`weekday_weekend`) is considered metadata about the transaction. This is why these columns were ignored when I converted the data to the required one-hot encoded format.

In [8]:
# Use get_dummies to one-hot encode the 'Item' column
one_hot_dataset = pd.get_dummies(bakery_dataset["Item"])

# Combine it with the 'Transaction' column
one_hot_dataset = pd.concat([bakery_dataset["Transaction"], one_hot_dataset], axis=1)
display(one_hot_dataset)

# Group by transaction and sum the one-hot encoded columns
# This counts the occurrences of each item in each transaction
encoded_dataset = one_hot_dataset.groupby('Transaction').sum()

# Convert counts to binary (0 or 1)
encoded_dataset = encoded_dataset.map(lambda x: 1 if x > 0 else 0)
display(encoded_dataset)

,Transaction,Adjustment,Afternoon with the baker,Alfajores,Argentina Night,Art Tray,Bacon,Baguette,Bakewell,Bare Popcorn,...,The BART,The Nomad,Tiffin,Toast,Truffles,Tshirt,Valentine's card,Vegan Feast,Vegan mincepie,Victorian Sponge
0,1,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,2,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,2,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,3,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,3,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20502,9682,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
20503,9682,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
20504,9683,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
20505,9683,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


,Adjustment,Afternoon with the baker,Alfajores,Argentina Night,Art Tray,Bacon,Baguette,Bakewell,Bare Popcorn,Basket,...,The BART,The Nomad,Tiffin,Toast,Truffles,Tshirt,Valentine's card,Vegan Feast,Vegan mincepie,Victorian Sponge
Transaction,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9680,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9681,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
9682,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Vary Dataset in Different Sizes based on Number of Unique Items
I will vary sizes on the number of unique items, but keeping the number of transactions does not change

In [9]:
# Define constants to vary dataset
num_transactions = 21 # Number of transactions in each dataset
num_datasets = 7 # Number of datasets
random_state = np.random.RandomState(seed=12071999) # To generate same sample everytime

result_datasets = []
temp_columns = set() # Flag to get different number of columns or unique items

In [10]:
# Generate new datasets
while len(temp_columns) < num_datasets:
    # Randomly sample 'num_transactions' rows from the original dataset
    sample_dataset = encoded_dataset.sample(n=num_transactions, random_state=random_state)

    # Identify and remove items (columns) that have no occurrences in this specific sample
    all_items = list(sample_dataset.sum().sort_values(ascending=False).items())
    item_names = [item_name for (item_name, count) in all_items if count > 0]

    # Create a new, cleaned dataset with only the columns that contain data
    new_dataset = sample_dataset[item_names]
    num_columns = len(new_dataset.columns)
    
    # Check if already created a dataset with this number of unique items
    if num_columns not in temp_columns:
        temp_columns.add(num_columns)
        result_datasets.append(new_dataset)

In [11]:
# Display new datasets
for (index, dataset) in enumerate(result_datasets):
    print("----- Dataset", index+1, "-----")
    print("Number of unique items:", dataset.shape[1])
    print("Number of transactions:", dataset.shape[0])
    display(dataset)

----- Dataset 1 -----
Number of unique items: 14
Number of transactions: 21


,Coffee,Bread,Cake,Cookies,Pastry,Farm House,Sandwich,Scone,Soup,Focaccia,Coke,Baguette,Vegan mincepie,Jam
Transaction,,,,,,,,,,,,,,
8187,0,0,0,1,0,0,0,0,0,0,0,1,0,0
2310,0,0,0,0,0,0,0,0,0,0,1,0,0,0
1770,0,0,0,0,0,1,0,0,0,0,0,0,0,0
7969,0,1,0,0,0,0,0,0,0,0,0,0,0,0
4356,0,0,0,0,0,1,0,1,0,0,0,0,0,0
7154,1,0,0,0,1,0,0,0,0,0,0,0,0,0
4898,0,1,0,0,0,0,0,0,0,0,0,0,0,0
2896,0,0,0,0,0,0,0,0,0,1,0,0,0,0
2331,1,0,0,0,1,0,0,0,0,0,0,0,0,0


----- Dataset 2 -----
Number of unique items: 16
Number of transactions: 21


,Coffee,Pastry,Toast,Bread,Cookies,Scandinavian,Tea,Cake,Sandwich,Brownie,Medialuna,Muffin,Farm House,Soup,Juice,Tartine
Transaction,,,,,,,,,,,,,,,,
5164,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
4008,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0
6686,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3244,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0
8265,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7416,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
8970,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
6835,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0
1321,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0


----- Dataset 3 -----
Number of unique items: 18
Number of transactions: 21


...

,Coffee,Bread,Sandwich,Tea,Hot chocolate,Truffles,Cake,Pastry,Fudge,Spanish Brunch,Chicken Stew,Coke,Alfajores,Duck egg,Mineral water,Toast,Jam
Transaction,,,,,,,,,,,,,,,,,
7017,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8892,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6006,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
6277,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
1183,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0
6236,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5169,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0
3110,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1
5124,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


----- Dataset 6 -----
Number of unique items: 23
Number of transactions: 21


,Coffee,Tea,Bread,Pastry,Cookies,Hot chocolate,Medialuna,Salad,Sandwich,Soup,...,Fudge,Smoothies,Spanish Brunch,Juice,The Nomad,Alfajores,Toast,Vegan mincepie,Baguette,Truffles
Transaction,,,,,,,,,,,,,,,,,,,,,
7106,1,0,0,0,0,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
6224,0,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2270,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1768,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1697,1,0,1,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9590,1,1,0,0,0,0,0,0,0,0,...,0,1,1,0,1,0,0,0,0,0
6049,0,0,1,0,0,0,0,0,0,0,...,1,0,0,0,0,1,0,0,0,0
6443,0,0,0,1,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,1,0


----- Dataset 7 -----
Number of unique items: 19
Number of transactions: 21


,Coffee,Bread,Toast,Cake,Tea,Soup,Hot chocolate,Pastry,Ella's Kitchen Pouches,Spanish Brunch,Sandwich,Scandinavian,Farm House,Fudge,Muffin,Scone,Cookies,Juice,Alfajores
Transaction,,,,,,,,,,,,,,,,,,,
3636,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
6562,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8772,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
2586,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1080,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
4572,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2555,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0
9437,0,0,0,0,1,1,0,0,0,1,0,0,0,0,1,0,1,0,0
9351,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## Orange 3 Association Rules Mining

Based on its GitHub codebase, Orange 3 uses the FP-Growth (Frequent Pattern Growth) algorithm for its association rules mining.
- Orange 3 associate FP-growth: https://github.com/biolab/orange3-associate/blob/master/orangecontrib/associate/fpgrowth.py

In [12]:
min_support = 0.04
min_confidence = 0.5

In [13]:
for (index, dataset) in enumerate(result_datasets):
    # Create a mapping from column index to item name
    mapping = {i: item_name for i, item_name in enumerate(dataset.columns)}

    # Find frequent itemsets
    start_orange_association_rules = time.time()
    itemsets = dict(frequent_itemsets(dataset.values, min_support=min_support))

    # Generate association rules from the frequent itemsets
    rules = list(association_rules(itemsets, min_confidence=min_confidence))
    end_orange_association_rules = time.time()

    # Present processed rules as results
    orange_results = []
    for antecedent, consequent, support, confidence in rules:
        antecedent_list = [mapping[item] for item in antecedent]
        consequent_list = [mapping[item] for item in consequent]

        orange_results.append({
            "Antecedent": ", ".join(antecedent_list),
            "Consequent": ", ".join(consequent_list),
            "Support": support,
            "Confidence": confidence
        })
    
    # Display results of orange 3 association rules mining 
    print("----- Dataset", index+1, "- Orange 3 Association Rules Mining -----")
    print("Number of unique items:", dataset.shape[1])
    print("Number of transactions:", dataset.shape[0])

    orange_association_rules_time = end_orange_association_rules - start_orange_association_rules
    print("Time:", orange_association_rules_time)
    orange_results_df = pd.DataFrame(orange_results)
    print("Results:")
    display(orange_results_df)

----- Dataset 1 - Orange 3 Association Rules Mining -----
Number of unique items: 14
Number of transactions: 21
Time: 0.00021409988403320312
Results:


,Antecedent,Consequent,Support,Confidence
0,"Bread, Cake",Coffee,1,1.0
1,"Coffee, Cake",Bread,1,0.5
2,Cake,"Coffee, Bread",1,0.5
3,"Coffee, Bread",Cake,1,0.5
4,Cake,Coffee,2,1.0
5,Cake,Bread,1,0.5
6,Cookies,Bread,1,0.5
7,Pastry,Coffee,2,1.0
8,Sandwich,Coffee,1,1.0
9,Scone,Farm House,1,1.0


----- Dataset 2 - Orange 3 Association Rules Mining -----
Number of unique items: 16
Number of transactions: 21
Time: 0.0007522106170654297
Results:


,Antecedent,Consequent,Support,Confidence
0,"Toast, Bread, Farm House",Coffee,1,1.0
1,"Coffee, Bread, Farm House",Toast,1,1.0
2,"Bread, Farm House","Coffee, Toast",1,1.0
3,"Coffee, Toast, Farm House",Bread,1,1.0
4,"Toast, Farm House","Coffee, Bread",1,1.0
...,...,...,...,...
71,Juice,Coffee,1,1.0
72,Juice,Toast,1,1.0
73,Juice,Soup,1,1.0
74,Soup,Juice,1,1.0


----- Dataset 3 - Orange 3 Association Rules Mining -----
Number of unique items: 18
Number of transactions: 21
Time: 0.0005340576171875
Results:


...

,Antecedent,Consequent,Support,Confidence
0,"Hot chocolate, Cake",Coffee,1,1.000000
1,"Coffee, Cake",Hot chocolate,1,1.000000
2,Cake,"Coffee, Hot chocolate",1,0.500000
3,"Coffee, Hot chocolate",Cake,1,1.000000
4,Hot chocolate,"Coffee, Cake",1,0.500000
5,"Alfajores, Hot chocolate",Tea,1,1.000000
6,"Tea, Hot chocolate",Alfajores,1,1.000000
7,Hot chocolate,"Tea, Alfajores",1,0.500000
8,Tea,"Alfajores, Hot chocolate",1,0.500000
9,"Tea, Alfajores",Hot chocolate,1,1.000000


----- Dataset 6 - Orange 3 Association Rules Mining -----
Number of unique items: 23
Number of transactions: 21
Time: 0.001177072525024414
Results:


,Antecedent,Consequent,Support,Confidence
0,"Bread, Cookies, Salad, Soup, Truffles",Tea,1,1.0
1,"Tea, Cookies, Salad, Soup, Truffles",Bread,1,1.0
2,"Soup, Cookies, Truffles, Salad","Tea, Bread",1,1.0
3,"Tea, Bread, Salad, Soup, Truffles",Cookies,1,1.0
4,"Soup, Bread, Truffles, Salad","Tea, Cookies",1,1.0
...,...,...,...,...
811,Cookies,Truffles,1,0.5
812,Salad,Truffles,1,0.5
813,Truffles,Salad,1,1.0
814,Truffles,Soup,1,1.0


----- Dataset 7 - Orange 3 Association Rules Mining -----
Number of unique items: 19
Number of transactions: 21
Time: 0.000446319580078125
Results:


,Antecedent,Consequent,Support,Confidence
0,"Spanish Brunch, Tea, Soup, Muffin",Cookies,1,1.0
1,"Tea, Soup, Muffin","Cookies, Spanish Brunch",1,1.0
2,"Soup, Muffin","Cookies, Spanish Brunch, Tea",1,1.0
3,"Tea, Muffin","Cookies, Spanish Brunch, Soup",1,1.0
4,Muffin,"Cookies, Spanish Brunch, Tea, Soup",1,1.0
...,...,...,...,...
239,Juice,Bread,1,1.0
240,Cake,Juice,1,0.5
241,Juice,Cake,1,1.0
242,Alfajores,Coffee,1,1.0


## Brute-Force Association Rules Mining

In [14]:
# Define brute-force algorithm function
def get_subsets(items):
    """
    Generates all non-empty subsets from a list of items.
    Example: get_subsets(['a', 'b']) -> {'a'}, {'b'}, {'a', 'b'}
    """
    return chain.from_iterable(combinations(items, r) for r in range(1, len(items) + 1))

def brute_force_association_rules(transactions, min_support, min_confidence):
    """
    Generates association rules using a brute force approach.

    Args:
        transactions (list of sets): The transaction database.
        min_support (float): The minimum support threshold.
        min_confidence (float): The minimum confidence threshold.

    Returns:
        list: A list of tuples, where each tuple represents a rule
              (antecedent, consequent, support, confidence).
    """
    unique_items = sorted(list(set(item for transaction in transactions for item in transaction)))
    all_itemsets = [frozenset(subset) for subset in get_subsets(unique_items)]
    
    num_transactions = len(transactions)
    
    # Step 1: Calculate Support for All Itemsets
    itemset_supports = {}
    for itemset in all_itemsets:
        count = 0
        for transaction in transactions:
            if itemset.issubset(transaction):
                count += 1
        support = count / num_transactions
        if support >= min_support:
            itemset_supports[itemset] = support

    final_rules = []

    # Step 2: Generate and Test All Rules
    for itemset, support in itemset_supports.items():
        if len(itemset) > 1:
            # Generate all possible rules from this itemset
            for antecedent in (frozenset(subset) for subset in get_subsets(itemset) if len(subset) < len(itemset)):
                consequent = itemset - antecedent
                
                # Check if the antecedent exists in our support dictionary
                if antecedent in itemset_supports:
                    antecedent_support = itemset_supports[antecedent]
                    confidence = support / antecedent_support
                    
                    if confidence >= min_confidence:
                        final_rules.append((antecedent, consequent, support, confidence))

    return final_rules

In [15]:
# Convert data into a list of sets
def convert_into_list_of_sets(data):
    transactions = []
    for index, row in data.iterrows():
        # For each row, get the column names where the value is 1
        itemset = set(row.index[row == 1])
        transactions.append(itemset)

    return transactions

for (index, dataset) in enumerate(result_datasets):
    transactions = convert_into_list_of_sets(dataset)

    start_brute_force_association_rules = time.time()
    rules = brute_force_association_rules(transactions, min_support=min_support, min_confidence=min_confidence)
    end_brute_force_association_rules = time.time()

    # Present processed rules as results
    brute_force_results = []
    for antecedent, consequent, support, confidence in rules:
        brute_force_results.append({
            "Antecedent": ", ".join(antecedent),
            "Consequent": ", ".join(consequent),
            "Support": support,
            "Confidence": confidence
        })
    
    # Display results of brute-force association rules mining 
    print("----- Dataset", index+1, "- Brute-Force Association Rules Mining -----")
    print("Number of unique items:", dataset.shape[1])
    print("Number of transactions:", dataset.shape[0])

    brute_force_association_rules_time = end_brute_force_association_rules - start_brute_force_association_rules
    print("Time:", brute_force_association_rules_time)
    brute_force_results_df = pd.DataFrame(brute_force_results)
    print("Results:")
    display(brute_force_results_df)

----- Dataset 1 - Brute-Force Association Rules Mining -----
Number of unique items: 14
Number of transactions: 21
Time: 0.019708871841430664
Results:


,Antecedent,Consequent,Support,Confidence
0,Baguette,Cookies,0.047619,1.0
1,Cookies,Baguette,0.047619,0.5
2,Cake,Bread,0.047619,0.5
3,Cookies,Bread,0.047619,0.5
4,Cake,Coffee,0.095238,1.0
5,Jam,Coffee,0.047619,1.0
6,Pastry,Coffee,0.095238,1.0
7,Sandwich,Coffee,0.047619,1.0
8,Vegan mincepie,Coffee,0.047619,1.0
9,Farm House,Scone,0.047619,0.5


----- Dataset 2 - Brute-Force Association Rules Mining -----
Number of unique items: 16
Number of transactions: 21
Time: 0.06974315643310547
Results:


,Antecedent,Consequent,Support,Confidence
0,Farm House,Bread,0.047619,1.0
1,Scandinavian,Bread,0.047619,0.5
2,Brownie,Cookies,0.047619,1.0
3,Cake,Coffee,0.047619,0.5
4,Cake,Pastry,0.047619,0.5
...,...,...,...,...
71,"Juice, Coffee","Soup, Toast",0.047619,1.0
72,"Soup, Juice, Toast",Coffee,0.047619,1.0
73,"Soup, Juice, Coffee",Toast,0.047619,1.0
74,"Soup, Toast, Coffee",Juice,0.047619,1.0


----- Dataset 3 - Brute-Force Association Rules Mining -----
Number of unique items: 18
Number of transactions: 21
Time: 0.35542821884155273
Results:


...

,Antecedent,Consequent,Support,Confidence
0,Hot chocolate,Alfajores,0.047619,0.500000
1,Alfajores,Hot chocolate,0.047619,1.000000
2,Alfajores,Tea,0.047619,1.000000
3,Tea,Alfajores,0.047619,0.500000
4,Cake,Coffee,0.047619,0.500000
5,Hot chocolate,Cake,0.047619,0.500000
6,Cake,Hot chocolate,0.047619,0.500000
7,Cake,Tea,0.047619,0.500000
8,Tea,Cake,0.047619,0.500000
9,Chicken Stew,Coffee,0.047619,1.000000


----- Dataset 6 - Brute-Force Association Rules Mining -----
Number of unique items: 23
Number of transactions: 21
Time: 16.707499027252197
Results:


,Antecedent,Consequent,Support,Confidence
0,Alfajores,Bread,0.047619,1.0
1,Fudge,Alfajores,0.047619,1.0
2,Alfajores,Fudge,0.047619,1.0
3,Baguette,Medialuna,0.047619,1.0
4,Medialuna,Baguette,0.047619,0.5
...,...,...,...,...
811,"Soup, Cookies, Tea, Bread, Truffles",Salad,0.047619,1.0
812,"Soup, Cookies, Tea, Salad, Truffles",Bread,0.047619,1.0
813,"Soup, Tea, Bread, Salad, Truffles",Cookies,0.047619,1.0
814,"Soup, Cookies, Bread, Salad, Truffles",Tea,0.047619,1.0


----- Dataset 7 - Brute-Force Association Rules Mining -----
Number of unique items: 19
Number of transactions: 21
Time: 0.43654704093933105
Results:


,Antecedent,Consequent,Support,Confidence
0,Alfajores,Bread,0.047619,1.0
1,Alfajores,Coffee,0.047619,1.0
2,Cake,Bread,0.047619,0.5
3,Ella's Kitchen Pouches,Bread,0.047619,1.0
4,Hot chocolate,Bread,0.047619,0.5
...,...,...,...,...
239,"Soup, Spanish Brunch, Cookies, Tea",Muffin,0.047619,1.0
240,"Soup, Spanish Brunch, Muffin, Tea",Cookies,0.047619,1.0
241,"Soup, Spanish Brunch, Muffin, Cookies",Tea,0.047619,1.0
242,"Soup, Muffin, Cookies, Tea",Spanish Brunch,0.047619,1.0


## Comparison between Orange 3 (FP-Growth) and Brute-Force Association Rules Mining